# Software FJ – Sistema Integral de Gestión de Clientes, Servicios y Reservas

**Curso:** Programación (213023) – UNAD
**Fase 4:** Componente práctico – Prácticas simuladas (manejo de excepciones)

Este cuaderno implementa un sistema orientado a objetos **sin base de datos**. Todo se guarda en objetos y listas en memoria; solo se usa un archivo para los **logs** (registro de eventos y errores).

**Qué cubre (según la guía y la rúbrica):**
1. Excepciones personalizadas, `try/except`, `try/except/else`, `try/except/finally` y encadenamiento (`raise ... from ...`).
2. Clase abstracta de entidades, `Cliente`, `Servicio` abstracto con 3 servicios hijos, y `Reserva`.
3. Métodos sobrecargados (costo con impuestos, descuentos y cupón).
4. Archivo de logs con todos los eventos y errores.
5. Más de 10 simulaciones (válidas e inválidas) donde el programa **nunca se detiene**.

## 1. Importaciones y configuración del archivo de logs

Se usa el módulo estándar `logging`. Cada evento se escribe en `logs_software_fj.log` con fecha, nivel y mensaje. El archivo se reinicia en cada ejecución (`mode="w"`) para que el log de la demostración quede limpio.

In [ ]:
import logging                      # Módulo estándar para registrar eventos y errores en un archivo
import re                           # Expresiones regulares para validar correo, teléfono y nombres
from abc import ABC, abstractmethod # Herramientas para crear clases y métodos abstractos

ARCHIVO_LOG = "logs_software_fj.log"  # Nombre del archivo donde se guardarán los logs


def configurar_logger():
    """Crea y devuelve el logger del sistema, escribiendo en ARCHIVO_LOG."""
    logger_fj = logging.getLogger("software_fj")                 # Obtiene (o crea) el logger con nombre propio
    logger_fj.setLevel(logging.DEBUG)                            # Registra todos los niveles (DEBUG, INFO, WARNING, ERROR)
    logger_fj.handlers.clear()                                   # Evita mensajes duplicados si se ejecuta la celda dos veces
    manejador = logging.FileHandler(ARCHIVO_LOG, mode="w", encoding="utf-8")  # Escribe en el archivo (UTF-8 para tildes)
    formato = logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s")  # Formato: fecha | nivel | mensaje
    manejador.setFormatter(formato)                              # Aplica el formato al manejador
    logger_fj.addHandler(manejador)                              # Conecta el manejador al logger
    return logger_fj                                             # Devuelve el logger listo para usar


logger = configurar_logger()                                     # Se crea el logger global del sistema
logger.info("Sistema Software FJ iniciado")                      # Primer evento registrado en el log

## 2. Excepciones personalizadas

Se crea una excepción base `ErrorSoftwareFJ` y excepciones hijas para cada tipo de problema (datos inválidos, parámetros faltantes, operaciones no permitidas, servicio no disponible, cálculos inconsistentes, reservas incorrectas). Tener una base común permite capturar **cualquier** error del sistema con un solo `except ErrorSoftwareFJ`.

In [ ]:
class ErrorSoftwareFJ(Exception):
    """Excepción base de todo el sistema Software FJ."""

    def __init__(self, mensaje, codigo="FJ-000"):
        super().__init__(mensaje)          # Guarda el mensaje en la clase Exception original
        self.mensaje = mensaje             # Mensaje legible del error
        self.codigo = codigo               # Código corto para identificar el tipo de error en el log

    def __str__(self):
        return f"[{self.codigo}] {self.mensaje}"  # Así se mostrará el error al imprimirlo


class ClienteInvalidoError(ErrorSoftwareFJ):
    """Datos del cliente incorrectos o cliente inexistente."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-101")


class ParametroInvalidoError(ErrorSoftwareFJ):
    """Un parámetro tiene un valor o tipo no permitido."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-102")


class ParametroFaltanteError(ErrorSoftwareFJ):
    """Falta un parámetro obligatorio al crear un objeto."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-103")


class ServicioNoDisponibleError(ErrorSoftwareFJ):
    """El servicio existe pero no está disponible para reservar."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-201")


class ReservaInvalidaError(ErrorSoftwareFJ):
    """Intento de reserva incorrecto."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-301")


class OperacionNoPermitidaError(ErrorSoftwareFJ):
    """Operación no válida para el estado actual (ej. cancelar dos veces)."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-302")


class CalculoInconsistenteError(ErrorSoftwareFJ):
    """El cálculo de costos produjo o recibió valores inconsistentes."""
    def __init__(self, mensaje):
        super().__init__(mensaje, "FJ-401")

## 3. Clase abstracta `EntidadBase` y clase `Cliente`

`EntidadBase` es la clase abstracta que representa **entidades generales** del sistema (cliente, servicio, reserva). Obliga a cada hija a implementar `describir()`.

`Cliente` encapsula sus datos personales (atributos privados con `_`) y los valida en los *setters*. Si un dato es inválido, lanza `ClienteInvalidoError` y el objeto **no se crea**.

In [ ]:
class EntidadBase(ABC):
    """Clase abstracta: representa cualquier entidad general del sistema."""

    _contador = 0  # Contador compartido para asignar identificadores únicos

    def __init__(self):
        EntidadBase._contador += 1         # Aumenta el contador global
        self._id = EntidadBase._contador   # Asigna el identificador a esta entidad

    @property
    def id(self):
        return self._id                    # Solo lectura: el id no se puede modificar desde fuera

    @abstractmethod
    def describir(self):
        """Cada entidad debe devolver una descripción en texto."""


class Cliente(EntidadBase):
    """Cliente de Software FJ con datos personales encapsulados y validados."""

    def __init__(self, nombre, documento, email, telefono):
        # Primero se valida (los setters lanzan error si algo está mal); así no se gasta un id con datos malos
        self.nombre = nombre               # Usa el setter de nombre
        self.documento = documento         # Usa el setter de documento
        self.email = email                 # Usa el setter de email
        self.telefono = telefono           # Usa el setter de teléfono
        super().__init__()                 # Solo si todo es válido se asigna el id

    # ----- nombre -----
    @property
    def nombre(self):
        return self._nombre                # Devuelve el nombre privado

    @nombre.setter
    def nombre(self, valor):
        if not isinstance(valor, str) or not valor.strip():           # Debe ser texto no vacío
            raise ClienteInvalidoError("El nombre es obligatorio y debe ser texto")
        valor = valor.strip()                                         # Quita espacios sobrantes
        if len(valor) < 3 or not re.fullmatch(r"[A-Za-zÁÉÍÓÚáéíóúÑñ ]+", valor):  # Mínimo 3 letras, sin números
            raise ClienteInvalidoError(f"Nombre inválido: '{valor}'")
        self._nombre = valor               # Guarda el valor ya validado

    # ----- documento -----
    @property
    def documento(self):
        return self._documento             # Devuelve el documento privado

    @documento.setter
    def documento(self, valor):
        valor = str(valor).strip()                                    # Convierte a texto y limpia
        if not re.fullmatch(r"\d{6,12}", valor):                      # Solo dígitos, de 6 a 12
            raise ClienteInvalidoError(f"Documento inválido: '{valor}' (6 a 12 dígitos)")
        self._documento = valor            # Guarda el documento validado

    # ----- email -----
    @property
    def email(self):
        return self._email                 # Devuelve el correo privado

    @email.setter
    def email(self, valor):
        valor = str(valor).strip().lower()                            # Normaliza el correo
        if not re.fullmatch(r"[\w.+-]+@[\w-]+(\.[\w-]+)+", valor):     # Formato básico usuario@dominio.ext
            raise ClienteInvalidoError(f"Correo inválido: '{valor}'")
        self._email = valor                # Guarda el correo validado

    # ----- teléfono -----
    @property
    def telefono(self):
        return self._telefono              # Devuelve el teléfono privado

    @telefono.setter
    def telefono(self, valor):
        valor = str(valor).strip()                                    # Convierte a texto y limpia
        if not re.fullmatch(r"3\d{9}", valor):                        # Celular colombiano: 10 dígitos que inician en 3
            raise ClienteInvalidoError(f"Teléfono inválido: '{valor}' (10 dígitos iniciando en 3)")
        self._telefono = valor             # Guarda el teléfono validado

    def describir(self):
        # Implementación obligatoria del método abstracto de EntidadBase
        return f"Cliente #{self.id}: {self.nombre} (doc. {self.documento}, {self.email}, {self.telefono})"

## 4. Clase abstracta `Servicio` y tres servicios especializados

`Servicio` define lo que **todo** servicio debe hacer: `calcular_costo()`, `describir()` y `validar_parametros()` (métodos abstractos, **sobrescritos** en cada hija → polimorfismo).

**Sobrecarga de métodos:** Python no tiene sobrecarga como Java, pero se logra con parámetros opcionales y revisión de tipos. `calcular_costo_total()` acepta cuatro variantes:

| Llamada | Qué calcula |
|---|---|
| `calcular_costo_total(horas)` | costo base |
| `calcular_costo_total(horas, 0.19)` | base + impuesto |
| `calcular_costo_total(horas, 0.19, 0.10)` | base − descuento + impuesto |
| `calcular_costo_total(horas, "FJ10")` | base − descuento por cupón |

In [ ]:
class Servicio(EntidadBase, ABC):
    """Clase abstracta de servicio: define el contrato y el cálculo de costos común."""

    CUPONES = {"FJ10": 0.10, "FJ20": 0.20}  # Cupones válidos y su porcentaje de descuento

    def __init__(self, nombre, tarifa_hora, disponible=True):
        if nombre is None or not str(nombre).strip():                 # El nombre es obligatorio
            raise ParametroFaltanteError("El servicio requiere un nombre")
        if isinstance(tarifa_hora, bool) or not isinstance(tarifa_hora, (int, float)) or tarifa_hora <= 0:
            raise ParametroInvalidoError(f"Tarifa por hora inválida: {tarifa_hora!r}")  # Debe ser número positivo
        self._nombre = str(nombre).strip()      # Nombre encapsulado
        self._tarifa_hora = float(tarifa_hora)  # Tarifa encapsulada
        self._disponible = bool(disponible)     # Disponibilidad encapsulada
        super().__init__()                      # Asigna el id (solo si los datos son válidos)

    # ----- propiedades de solo lectura / controlada -----
    @property
    def nombre(self):
        return self._nombre

    @property
    def tarifa_hora(self):
        return self._tarifa_hora

    @property
    def disponible(self):
        return self._disponible

    def cambiar_disponibilidad(self, estado):
        self._disponible = bool(estado)                               # Permite activar o desactivar el servicio
        logger.info(f"Servicio #{self.id} '{self.nombre}' disponible={self._disponible}")

    # ----- métodos abstractos (cada hija los sobrescribe) -----
    @abstractmethod
    def calcular_costo(self, horas):
        """Costo base del servicio para una cantidad de horas."""

    @abstractmethod
    def validar_parametros(self):
        """Valida los datos propios de cada tipo de servicio."""

    # describir() sigue abstracto (viene de EntidadBase) y también se sobrescribe en cada hija

    # ----- método sobrecargado -----
    def calcular_costo_total(self, horas, *args):
        """Sobrecarga simulada: (horas) | (horas, impuesto) | (horas, impuesto, descuento) | (horas, cupon)."""
        base = self.calcular_costo(horas)       # Costo base según el tipo de servicio (polimorfismo)
        impuesto, descuento = 0.0, 0.0          # Valores por defecto: sin impuesto ni descuento

        if len(args) == 1 and isinstance(args[0], str):               # Variante con cupón de texto
            cupon = args[0].upper()
            if cupon not in self.CUPONES:                             # Cupón inexistente
                raise ParametroInvalidoError(f"Cupón no válido: '{args[0]}'")
            descuento = self.CUPONES[cupon]                           # Toma el descuento del cupón
        elif len(args) == 1:                                          # Variante con impuesto
            impuesto = args[0]
        elif len(args) == 2:                                          # Variante con impuesto y descuento
            impuesto, descuento = args
        elif len(args) > 2:                                           # Demasiados parámetros
            raise ParametroInvalidoError("calcular_costo_total admite máximo 2 parámetros extra")

        for valor in (impuesto, descuento):                           # Validar que sean números
            if isinstance(valor, bool) or not isinstance(valor, (int, float)):
                raise CalculoInconsistenteError(f"Valor no numérico en el cálculo: {valor!r}")
        if descuento < 0 or descuento > 1:                            # Un descuento debe estar entre 0 % y 100 %
            raise CalculoInconsistenteError(f"Descuento fuera de rango: {descuento}")
        if impuesto < 0 or impuesto > 1:                              # Un impuesto debe estar entre 0 % y 100 %
            raise CalculoInconsistenteError(f"Impuesto fuera de rango: {impuesto}")

        total = base * (1 - descuento) * (1 + impuesto)               # Fórmula final del costo
        if total <= 0:                                                # Un costo no puede ser cero ni negativo
            raise CalculoInconsistenteError(f"Costo total inconsistente: {total}")
        return round(total, 2)                                        # Redondea a 2 decimales


def validar_horas(horas, maximo):
    """Función auxiliar: valida que las horas sean un entero entre 1 y el máximo del servicio."""
    if isinstance(horas, bool) or not isinstance(horas, int):         # Debe ser entero
        raise ParametroInvalidoError(f"Las horas deben ser un número entero, recibido: {horas!r}")
    if horas < 1 or horas > maximo:                                   # Y estar dentro del rango permitido
        raise ParametroInvalidoError(f"Horas fuera de rango (1 a {maximo}): {horas}")


class ReservaSala(Servicio):
    """Servicio 1: reserva de salas de reuniones."""

    def __init__(self, nombre, tarifa_hora, capacidad, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)             # Datos comunes
        self._capacidad = capacidad                                   # Dato propio: personas que caben
        self.validar_parametros()                                     # Valida el dato propio

    def validar_parametros(self):
        if isinstance(self._capacidad, bool) or not isinstance(self._capacidad, int) or self._capacidad <= 0:
            raise ParametroInvalidoError(f"Capacidad inválida: {self._capacidad!r}")

    def calcular_costo(self, horas):
        validar_horas(horas, 12)                                      # Máximo 12 horas por reserva
        costo = self.tarifa_hora * horas                              # Tarifa x horas
        if self._capacidad > 20:                                      # Recargo del 10 % por salas grandes
            costo *= 1.10
        return costo

    def describir(self):
        return f"Sala #{self.id} '{self.nombre}' – capacidad {self._capacidad} personas – ${self.tarifa_hora:,.0f}/h"


class AlquilerEquipo(Servicio):
    """Servicio 2: alquiler de equipos tecnológicos."""

    def __init__(self, nombre, tarifa_hora, tipo_equipo, seguro=15000, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)             # Datos comunes
        self._tipo_equipo = tipo_equipo                               # Dato propio: tipo de equipo
        self._seguro = seguro                                         # Dato propio: seguro fijo (parámetro opcional)
        self.validar_parametros()                                     # Valida los datos propios

    def validar_parametros(self):
        if not self._tipo_equipo or not str(self._tipo_equipo).strip():
            raise ParametroFaltanteError("El alquiler requiere indicar el tipo de equipo")
        if isinstance(self._seguro, bool) or not isinstance(self._seguro, (int, float)) or self._seguro < 0:
            raise ParametroInvalidoError(f"Seguro inválido: {self._seguro!r}")

    def calcular_costo(self, horas):
        validar_horas(horas, 72)                                      # Máximo 72 horas (3 días)
        return self.tarifa_hora * horas + self._seguro                # Tarifa x horas + seguro fijo

    def describir(self):
        return f"Equipo #{self.id} '{self.nombre}' ({self._tipo_equipo}) – ${self.tarifa_hora:,.0f}/h + seguro ${self._seguro:,.0f}"


class AsesoriaEspecializada(Servicio):
    """Servicio 3: asesorías con especialistas."""

    NIVELES = {"junior": 1.0, "senior": 1.5, "experto": 2.0}          # Multiplicador de tarifa según nivel

    def __init__(self, nombre, tarifa_hora, nivel, disponible=True):
        super().__init__(nombre, tarifa_hora, disponible)             # Datos comunes
        self._nivel = str(nivel).lower() if nivel is not None else "" # Dato propio: nivel del asesor
        self.validar_parametros()                                     # Valida el dato propio

    def validar_parametros(self):
        if self._nivel not in self.NIVELES:                           # Solo junior, senior o experto
            raise ParametroInvalidoError(f"Nivel de asesor inválido: '{self._nivel}'")

    def calcular_costo(self, horas):
        validar_horas(horas, 8)                                       # Máximo 8 horas por asesoría
        return self.tarifa_hora * horas * self.NIVELES[self._nivel]   # Tarifa x horas x multiplicador

    def describir(self):
        return f"Asesoría #{self.id} '{self.nombre}' – nivel {self._nivel} – ${self.tarifa_hora:,.0f}/h"

## 5. Clase `Reserva`

Integra **cliente + servicio + duración + estado**. Estados posibles: `PENDIENTE`, `CONFIRMADA`, `CANCELADA`, `FALLIDA`.

- `confirmar()` y `cancelar()` validan que la operación sea permitida.
- `procesar()` usa **`try / except / else / finally`** y **encadenamiento de excepciones** (`raise ... from error`), que conserva el error original como causa (`__cause__`).

In [ ]:
class Reserva(EntidadBase):
    """Reserva que une un cliente, un servicio, una duración y un estado."""

    def __init__(self, cliente, servicio, horas):
        if not isinstance(cliente, Cliente):                          # El cliente debe ser un objeto Cliente
            raise ReservaInvalidaError("La reserva requiere un cliente válido")
        if not isinstance(servicio, Servicio):                        # El servicio debe ser un objeto Servicio
            raise ReservaInvalidaError("La reserva requiere un servicio válido")
        self._cliente = cliente        # Cliente que reserva
        self._servicio = servicio      # Servicio reservado
        self._horas = horas            # Duración solicitada (se valida al calcular el costo)
        self._estado = "PENDIENTE"     # Estado inicial
        self._costo_total = None       # Se calcula al procesar
        super().__init__()             # Asigna el id
        logger.info(f"Reserva #{self.id} creada: {cliente.nombre} -> {servicio.nombre} ({horas} h)")

    @property
    def estado(self):
        return self._estado            # Estado actual (solo lectura)

    @property
    def costo_total(self):
        return self._costo_total       # Costo calculado (solo lectura)

    @property
    def cliente(self):
        return self._cliente

    @property
    def servicio(self):
        return self._servicio

    def confirmar(self):
        """Confirma la reserva si está pendiente y el servicio está disponible."""
        if self._estado != "PENDIENTE":                               # Solo se confirma lo pendiente
            raise OperacionNoPermitidaError(f"No se puede confirmar una reserva en estado {self._estado}")
        if not self._servicio.disponible:                             # Servicio no disponible
            raise ServicioNoDisponibleError(f"El servicio '{self._servicio.nombre}' no está disponible")
        self._estado = "CONFIRMADA"                                   # Cambia el estado
        logger.info(f"Reserva #{self.id} confirmada")

    def cancelar(self, motivo="Sin motivo"):
        """Cancela la reserva si aún no está cancelada."""
        if self._estado == "CANCELADA":                               # No se cancela dos veces
            raise OperacionNoPermitidaError(f"La reserva #{self.id} ya estaba cancelada")
        if self._estado == "FALLIDA":                                 # Una reserva fallida no se cancela
            raise OperacionNoPermitidaError(f"La reserva #{self.id} fallida no se puede cancelar")
        self._estado = "CANCELADA"                                    # Cambia el estado
        logger.warning(f"Reserva #{self.id} cancelada. Motivo: {motivo}")

    def procesar(self, *parametros_costo):
        """Confirma y calcula el costo. Usa try/except/else/finally y encadenamiento de excepciones."""
        try:
            self.confirmar()                                                       # 1) Intenta confirmar
            costo = self._servicio.calcular_costo_total(self._horas, *parametros_costo)  # 2) Calcula (método sobrecargado)
        except (ServicioNoDisponibleError, OperacionNoPermitidaError) as error:
            self._estado = "FALLIDA" if self._estado == "PENDIENTE" else self._estado  # Marca fallida si seguía pendiente
            # Encadenamiento: se lanza un error de más alto nivel conservando el original como causa
            raise ReservaInvalidaError(f"No se pudo procesar la reserva #{self.id}") from error
        except (ParametroInvalidoError, CalculoInconsistenteError) as error:
            self._estado = "FALLIDA"                                               # El cálculo falló: la reserva queda fallida
            raise ReservaInvalidaError(f"Cálculo inválido en la reserva #{self.id}") from error
        except (TypeError, ValueError, ZeroDivisionError) as error:
            self._estado = "FALLIDA"                                               # Errores nativos de Python
            raise CalculoInconsistenteError(f"Error inesperado de cálculo en la reserva #{self.id}") from error
        else:
            # El bloque else solo se ejecuta si NO hubo ninguna excepción
            self._costo_total = costo
            logger.info(f"Reserva #{self.id} procesada. Costo total: ${costo:,.2f}")
            return costo
        finally:
            # El bloque finally se ejecuta SIEMPRE (haya error o no)
            logger.debug(f"Fin del procesamiento de la reserva #{self.id} (estado: {self._estado})")

    def describir(self):
        costo = f"${self._costo_total:,.2f}" if self._costo_total is not None else "sin calcular"
        return (f"Reserva #{self.id}: {self._cliente.nombre} | {self._servicio.nombre} | "
                f"{self._horas} h | {self._estado} | {costo}")

## 6. Clase `SistemaFJ` (gestor)

Administra las listas internas de clientes, servicios y reservas. **Cada operación captura sus propios errores**, los registra en el log y devuelve `None` en vez de detener el programa. Esto es lo que mantiene el sistema estable.

También muestra el uso de:
- `try/except/else/finally` en cada método,
- encadenamiento con `raise ... from` cuando faltan parámetros (`TypeError` → `ParametroFaltanteError`).

In [ ]:
class SistemaFJ:
    """Gestor general: maneja listas internas (sin base de datos) y protege el programa de errores."""

    TIPOS_SERVICIO = {                       # Relaciona el nombre del tipo con su clase
        "sala": ReservaSala,
        "equipo": AlquilerEquipo,
        "asesoria": AsesoriaEspecializada,
    }

    def __init__(self):
        self.clientes = []                   # Lista interna de clientes
        self.servicios = []                  # Lista interna de servicios
        self.reservas = []                   # Lista interna de reservas
        self.errores = 0                     # Contador de errores controlados
        self.exitos = 0                      # Contador de operaciones exitosas

    def _registrar_error(self, contexto, error):
        """Registra un error en el log (incluyendo su causa si fue encadenado) y lo cuenta."""
        self.errores += 1
        causa = f" | causa: {error.__cause__!r}" if error.__cause__ else ""  # Muestra el error original si existe
        logger.error(f"{contexto}: {error}{causa}")
        print(f"   ✗ ERROR controlado -> {error}{causa}")

    def registrar_cliente(self, **datos):
        """Crea un cliente y lo guarda en la lista. Devuelve el cliente o None si hubo error."""
        try:
            try:
                cliente = Cliente(**datos)                            # Intenta crear el cliente
            except TypeError as error:                                # Faltan o sobran argumentos
                raise ParametroFaltanteError(f"Datos de cliente incompletos: {error}") from error  # Encadenamiento
        except ErrorSoftwareFJ as error:
            self._registrar_error("registrar_cliente", error)         # Error propio del sistema
            return None
        except Exception as error:                                    # Red de seguridad: cualquier otro error
            self._registrar_error("registrar_cliente (inesperado)", error)
            return None
        else:
            self.clientes.append(cliente)                             # Solo si todo salió bien se guarda
            self.exitos += 1
            logger.info(f"Cliente registrado: {cliente.describir()}")
            print(f"   ✓ {cliente.describir()}")
            return cliente
        finally:
            logger.debug("Fin de registrar_cliente")                  # Se ejecuta siempre

    def crear_servicio(self, tipo, **datos):
        """Crea un servicio del tipo indicado ('sala', 'equipo', 'asesoria'). Devuelve el servicio o None."""
        try:
            clase = self.TIPOS_SERVICIO.get(str(tipo).lower())        # Busca la clase según el tipo
            if clase is None:                                         # Tipo inexistente
                raise ParametroInvalidoError(f"Tipo de servicio desconocido: '{tipo}'")
            try:
                servicio = clase(**datos)                             # Crea el servicio (polimorfismo)
            except TypeError as error:                                # Faltan parámetros
                raise ParametroFaltanteError(f"Parámetros de servicio incompletos: {error}") from error
        except ErrorSoftwareFJ as error:
            self._registrar_error("crear_servicio", error)
            return None
        except Exception as error:
            self._registrar_error("crear_servicio (inesperado)", error)
            return None
        else:
            self.servicios.append(servicio)
            self.exitos += 1
            logger.info(f"Servicio creado: {servicio.describir()}")
            print(f"   ✓ {servicio.describir()}")
            return servicio
        finally:
            logger.debug("Fin de crear_servicio")

    def crear_reserva(self, cliente, servicio, horas, *parametros_costo, cancelar_despues=False):
        """Crea y procesa una reserva. Devuelve la reserva o None si falló."""
        reserva = None                                                # Se declara antes para usarla en finally
        try:
            reserva = Reserva(cliente, servicio, horas)               # Crea la reserva
            self.reservas.append(reserva)                             # La guarda aunque luego falle (queda como FALLIDA)
            reserva.procesar(*parametros_costo)                       # Confirma y calcula el costo
            if cancelar_despues:                                      # Opción para simular cancelaciones
                reserva.cancelar("Solicitud del cliente")
        except ErrorSoftwareFJ as error:
            self._registrar_error("crear_reserva", error)
            return None
        except Exception as error:
            self._registrar_error("crear_reserva (inesperado)", error)
            return None
        else:
            self.exitos += 1
            print(f"   ✓ {reserva.describir()}")
            return reserva
        finally:
            estado = reserva.estado if reserva else "NO CREADA"
            logger.debug(f"Fin de crear_reserva (estado: {estado})")

    def cancelar_reserva(self, reserva, motivo="Sin motivo"):
        """Cancela una reserva existente. Devuelve True o False."""
        try:
            reserva.cancelar(motivo)                                  # Intenta cancelar
        except ErrorSoftwareFJ as error:
            self._registrar_error("cancelar_reserva", error)
            return False
        except AttributeError as error:                               # Se pasó algo que no es una reserva (ej. None)
            self._registrar_error("cancelar_reserva (objeto inválido)", error)
            return False
        else:
            self.exitos += 1
            print(f"   ✓ {reserva.describir()}")
            return True
        finally:
            logger.debug("Fin de cancelar_reserva")

    def buscar_cliente(self, documento):
        """Busca un cliente por documento. Lanza ClienteInvalidoError si no existe."""
        for cliente in self.clientes:                                 # Recorre la lista interna
            if cliente.documento == str(documento):
                return cliente
        raise ClienteInvalidoError(f"No existe un cliente con documento {documento}")

## 7. Simulación de 15 operaciones (válidas e inválidas)

Cada simulación se ejecuta dentro de la función `ejecutar()`, que tiene su propio `try/except`. Aunque ocurra un error grave, el ciclo **continúa** con la siguiente operación. Al final se muestra un resumen y el contenido del archivo de logs.

In [ ]:
sistema = SistemaFJ()          # Se crea el sistema (todo en memoria, sin base de datos)
contador_sim = 0               # Numera las simulaciones


def ejecutar(descripcion, accion):
    """Ejecuta una simulación protegida: ningún error puede detener el programa."""
    global contador_sim
    contador_sim += 1
    print(f"\n[{contador_sim:02d}] {descripcion}")
    logger.info(f"--- Simulación {contador_sim:02d}: {descripcion}")
    try:
        return accion()                                               # Ejecuta la operación
    except Exception as error:                                        # Última red de seguridad
        logger.critical(f"Error no previsto en simulación {contador_sim}: {error!r}")
        print(f"   ✗ ERROR no previsto controlado -> {error!r}")
        return None


# ---------- CLIENTES ----------
ana = ejecutar("Cliente válido: Ana Pérez",
               lambda: sistema.registrar_cliente(nombre="Ana Pérez", documento="1098765432",
                                                 email="ana.perez@correo.com", telefono="3101234567"))
luis = ejecutar("Cliente válido: Luis Gómez",
                lambda: sistema.registrar_cliente(nombre="Luis Gómez", documento="80123456",
                                                  email="luis.gomez@empresa.co", telefono="3209998877"))
ejecutar("Cliente INVÁLIDO: correo sin @",
         lambda: sistema.registrar_cliente(nombre="Carlos Ruiz", documento="1234567",
                                           email="carlos.correo.com", telefono="3001112233"))
ejecutar("Cliente INVÁLIDO: documento con letras y teléfono corto",
         lambda: sistema.registrar_cliente(nombre="María Díaz", documento="12AB56",
                                           email="maria@correo.com", telefono="300"))
ejecutar("Cliente con PARÁMETRO FALTANTE (sin email ni teléfono)",
         lambda: sistema.registrar_cliente(nombre="Pedro Mora", documento="7654321"))

# ---------- SERVICIOS ----------
sala = ejecutar("Servicio válido: Sala de juntas (capacidad 25)",
                lambda: sistema.crear_servicio("sala", nombre="Sala Ejecutiva", tarifa_hora=50000, capacidad=25))
equipo = ejecutar("Servicio válido: Alquiler de portátil",
                  lambda: sistema.crear_servicio("equipo", nombre="Portátil Pro", tarifa_hora=30000,
                                                 tipo_equipo="Portátil"))
asesoria = ejecutar("Servicio válido: Asesoría en ciberseguridad (experto)",
                    lambda: sistema.crear_servicio("asesoria", nombre="Asesoría Ciberseguridad",
                                                   tarifa_hora=120000, nivel="experto"))
ejecutar("Servicio INVÁLIDO: tarifa negativa",
         lambda: sistema.crear_servicio("sala", nombre="Sala Fantasma", tarifa_hora=-500, capacidad=10))
ejecutar("Servicio INVÁLIDO: tipo inexistente",
         lambda: sistema.crear_servicio("helicoptero", nombre="Vuelo", tarifa_hora=1000))

# ---------- RESERVAS ----------
ejecutar("Reserva EXITOSA: sala 3 h con IVA 19 % y descuento 10 %",
         lambda: sistema.crear_reserva(ana, sala, 3, 0.19, 0.10))
ejecutar("Reserva EXITOSA: asesoría 2 h con cupón FJ20",
         lambda: sistema.crear_reserva(luis, asesoria, 2, "FJ20"))
ejecutar("Reserva FALLIDA: horas excesivas (asesoría de 20 h)",
         lambda: sistema.crear_reserva(ana, asesoria, 20))
ejecutar("Reserva FALLIDA: cupón inexistente",
         lambda: sistema.crear_reserva(luis, equipo, 5, "PROMO99"))
ejecutar("Reserva FALLIDA: descuento del 150 % (cálculo inconsistente)",
         lambda: sistema.crear_reserva(ana, equipo, 4, 0.19, 1.5))

equipo.cambiar_disponibilidad(False)                                   # Se desactiva el equipo para la siguiente prueba
ejecutar("Reserva FALLIDA: servicio no disponible",
         lambda: sistema.crear_reserva(luis, equipo, 2))
ejecutar("Reserva FALLIDA: cliente inexistente (None)",
         lambda: sistema.crear_reserva(None, sala, 2))

reserva_cancelable = ejecutar("Reserva EXITOSA y cancelada después",
                              lambda: sistema.crear_reserva(ana, sala, 1, 0.19, cancelar_despues=True))
ejecutar("Cancelación INVÁLIDA: la reserva ya estaba cancelada",
         lambda: sistema.cancelar_reserva(reserva_cancelable, "Intento repetido"))
ejecutar("Búsqueda de cliente inexistente (error no capturado dentro del método)",
         lambda: sistema.buscar_cliente("99999999"))

In [ ]:
# ---------- RESUMEN FINAL ----------
print("\n" + "=" * 70)
print("RESUMEN DE LA SIMULACIÓN")
print("=" * 70)
print(f"Simulaciones ejecutadas : {contador_sim}")
print(f"Operaciones exitosas    : {sistema.exitos}")
print(f"Errores controlados     : {sistema.errores}")
print(f"Clientes registrados    : {len(sistema.clientes)}")
print(f"Servicios creados       : {len(sistema.servicios)}")
print(f"Reservas registradas    : {len(sistema.reservas)}")
print("\nEstado de las reservas:")
for r in sistema.reservas:
    print("  -", r.describir())
print("\n✔ El programa terminó todas las simulaciones sin detenerse.")
logger.info("Simulación finalizada: el sistema continuó estable")

## 8. Contenido del archivo de logs

Aquí se lee el archivo `logs_software_fj.log` para comprobar que **todos los eventos y errores** quedaron registrados.

In [ ]:
for manejador in logger.handlers:      # Se fuerza la escritura pendiente al disco
    manejador.flush()

with open(ARCHIVO_LOG, "r", encoding="utf-8") as archivo:   # Abre el log en modo lectura
    contenido = archivo.read()                              # Lee todo el contenido
print(contenido)                                            # Lo muestra en pantalla
print(f"Total de líneas en el log: {len(contenido.splitlines())}")
print(f"Errores registrados (ERROR/CRITICAL): {contenido.count('| ERROR') + contenido.count('| CRITICAL')}")

## 9. Conclusiones técnicas

- Las **excepciones personalizadas** con una clase base común permiten capturar y clasificar cualquier error del dominio.
- `try/except/else/finally` separa claramente el camino de error, el camino exitoso y la limpieza final.
- El **encadenamiento** (`raise ... from error`) conserva la causa original y facilita el diagnóstico en el log.
- La **abstracción, herencia y polimorfismo** permiten agregar nuevos servicios sin modificar `Reserva` ni `SistemaFJ`.
- El **archivo de logs** deja trazabilidad completa y el programa se mantiene estable ante datos inválidos.